In [38]:
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [39]:
np.random.seed(42)
tf.random.set_seed(42)

In [40]:
data = pd.read_csv("Dataset/creditcard.csv")
print("Shape:", data.shape)

Shape: (284807, 31)


In [41]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [42]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [44]:
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0])

Series([], dtype: int64)


In [45]:
duplicate_count = data.duplicated().sum()
print("Duplicate rows:", duplicate_count)
data = data.drop_duplicates()
print("Shape after removing duplicates:", data.shape)

Duplicate rows: 1081
Shape after removing duplicates: (283726, 31)


In [46]:
print(data["Class"].value_counts())

Class
0    283253
1       473
Name: count, dtype: int64


In [47]:
class_counts = data["Class"].value_counts()

normal_count = class_counts.get(0, 0)
fraud_count = class_counts.get(1, 0)

total = len(data)

normal_percentage = normal_count / total * 100
fraud_percentage = fraud_count / total * 100

print(f"Normal transactions: {normal_count}")
print(f"Fraud transactions: {fraud_count}")
print(f"Normal percentage: {normal_percentage:.4f}%")
print(f"Fraud percentage: {fraud_percentage:.4f}%")

Normal transactions: 283253
Fraud transactions: 473
Normal percentage: 99.8333%
Fraud percentage: 0.1667%


In [48]:
X = data.drop("Class", axis=1)
y = data["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (283726, 30)
y shape: (283726,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (226980, 30)
X_test: (56746, 30)
y_train: (226980,)
y_test: (56746,)


In [49]:
print("Training distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

stratify=y

Training distribution:
Class
0    226602
1       378
Name: count, dtype: int64

Testing distribution:
Class
0    56651
1       95
Name: count, dtype: int64


In [50]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [51]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Scaler saved successfully!")

Scaler saved successfully!


In [52]:
model = Sequential(
    [
        Dense(128, activation="relu", input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [53]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │         3,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,337 (56.00 KB)

 Trainable params: 14,337 (56.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

In [ ]:
import os

log_dir = os.path.join("logs", "fit", datetime.now().strftime("%Y%m%d-%H%M%S"))

print("TensorBoard log directory:")
print(log_dir)

TensorBoard log directory:
logs/fit/20260902-224113


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train
)

class_weights = dict(zip(classes, class_weights_array))

print("Class weights:")
print(class_weights)

Class weights:
{np.int64(0): np.float64(0.5008340614822464), np.int64(1): np.float64(300.23809523809524)}


In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stopping, tensorboard_callback],
    verbose=1,
)

Epoch 1/30
710/710 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.9618 - loss: 0.3113 - precision: 0.0353 - recall: 0.8400 - val_accuracy: 0.9697 - val_loss: 0.1391 - val_precision: 0.0512 - val_recall: 0.9487
Epoch 2/30
710/710 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9662 - loss: 0.1650 - precision: 0.0430 - recall: 0.9167 - val_accuracy: 0.9813 - val_loss: 0.0883 - val_precision: 0.0796 - val_recall: 0.9359
Epoch 3/30
710/710 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9752 - loss: 0.1282 - precision: 0.0588 - recall: 0.9333 - val_accuracy: 0.9797 - val_loss: 0.0921 - val_precision: 0.0728 - val_recall: 0.9231
Epoch 4/30
710/710 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9771 - loss: 0.1196 - precision: 0.0640 - recall: 0.9433 - val_accuracy: 0.9811 - val_loss: 0.0871 - val_precision: 0.0778 - val_recall: 0.9231
Epoch 5/30
710/710 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9688 - loss: 0.1247 - precision: 0.0471 - recall: 0.9300 - val_accuracy: 0.9842 - val_lo

In [ ]:
evaluation = model.evaluate(X_test_scaled, y_test, verbose=1)

print("Evaluation Results:")
for name, value in zip(model.metrics_names, evaluation):
    print(f"{name}: {value:.6f}")

1774/1774 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.3720 - loss: 0.7498 - precision: 7.5858e-04 - recall: 0.2842
Evaluation Results:
loss: 0.749763
compile_metrics: 0.372044


In [59]:
y_probability = model.predict(X_test_scaled, verbose=1).ravel()
print("Probability shape:", y_probability.shape)

1774/1774 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Probability shape: (56746,)


In [60]:
threshold = 0.5
y_pred = (y_probability >= threshold).astype(int)

In [35]:
BEST_THRESHOLD = 0.5

print("Selected threshold:", BEST_THRESHOLD)

Selected threshold: 0.5


In [36]:
model.save("model.keras")

print("Model saved successfully!")

Model saved successfully!


In [61]:
sample = X_test.iloc[0:1]
sample_scaled = scaler.transform(sample)
sample_probability = model.predict(sample_scaled, verbose=0)[0][0]
sample_prediction = int(sample_probability >= BEST_THRESHOLD)

print(f"Fraud probability: " f"{sample_probability * 100:.4f}%")
if sample_prediction == 1:
    print("Prediction: FRAUD")
else:
    print("Prediction: NORMAL")
print("Actual:", "FRAUD" if y_test.iloc[0] == 1 else "NORMAL")

Fraud probability: 47.3937%
Prediction: NORMAL
Actual: NORMAL
